## Notebook summary

| Item | Details |
| --- | --- |
| Purpose | 01 - Original-Image Base Training |
| Model / workflow | SE-ResNeXt-50 32x4d |
| Input | 384x384 published crops (kneeKL224 source) |
| Loss | Cross-Entropy (CE) |
| Training / pipeline | Base training, full network |
| Result | (filled in after the run) |

# 01 - Original-Image Base Training (SE-ResNeXt-50 32x4d)

Trains the base CE checkpoint on the **published** (pre-cropped) images. Notebook 02 adapts this
checkpoint to production YOLO ROIs, so run this one first.

- input: published crops, resized to 384x384
- loss: cross-entropy only
- sampler: inverse-frequency `WeightedRandomSampler`, so Grade 4 is not drowned out
- selection: validation `0.55*QWK + 0.30*macroF1 + 0.15*macroAP`
- the **test** split is never touched here

On completion the run directory is written to `SELECTED_CHECKPOINT.txt`, which Notebook 02 reads.
That pointer is what stops a stale run timestamp being hard-coded into the next stage.

In [1]:
!pip -q install "timm>=1.0" "h5py>=3.9"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import json
import random
from datetime import datetime, timezone
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import average_precision_score, cohen_kappa_score, precision_recall_fscore_support
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms
from tqdm.auto import tqdm

# ---- Focal CORN helpers (inlined for self-containment) -----------------
# See notebooks/_focal_corn_helpers.py for the canonical source.
NUM_CLASSES = 5
NUM_TASKS = NUM_CLASSES - 1
TASK_WEIGHTS = (1.0, 1.2, 2.0, 3.5)
FOCAL_GAMMA = 2.0
FOCAL_ALPHA = 0.25
LABEL_SMOOTHING = 0.10

def corn_loss(logits, y_train, num_classes=NUM_CLASSES, task_weights=TASK_WEIGHTS):
    loss = 0.0
    for k in range(num_classes - 1):
        mask = y_train >= k
        if not mask.any():
            continue
        logits_k = logits[mask, k]
        targets_k = (y_train[mask] > k).float()
        targets_k = targets_k * (1 - LABEL_SMOOTHING) + (1 - targets_k) * LABEL_SMOOTHING
        w_k = task_weights[k] if k < len(task_weights) else 1.0
        loss = loss + w_k * F.binary_cross_entropy_with_logits(logits_k, targets_k)
    return loss / (num_classes - 1)

def focal_corn_loss(logits, y_train, num_classes=NUM_CLASSES, gamma=FOCAL_GAMMA, alpha=FOCAL_ALPHA):
    loss = 0.0
    for k in range(num_classes - 1):
        mask = y_train >= k
        if not mask.any():
            continue
        logits_k = logits[mask, k]
        targets_k = (y_train[mask] > k).float()
        targets_k = targets_k * (1 - LABEL_SMOOTHING) + (1 - targets_k) * LABEL_SMOOTHING
        bce = F.binary_cross_entropy_with_logits(logits_k, targets_k, reduction="none")
        p = torch.sigmoid(logits_k)
        p_t = p * targets_k + (1 - p) * (1 - targets_k)
        focal_weight = alpha * (1 - p_t) ** gamma
        loss = loss + (focal_weight * bce).mean()
    return loss / (num_classes - 1)

def corn_probas(logits):
    cond_probas = torch.sigmoid(logits)
    batch_size = logits.size(0)
    num_classes = logits.size(1) + 1
    probas = torch.zeros(batch_size, num_classes, device=logits.device)
    cumprod = torch.cumprod(cond_probas, dim=1)
    probas[:, 0] = 1.0 - cond_probas[:, 0]
    for i in range(1, num_classes - 1):
        probas[:, i] = cumprod[:, i - 1] * (1.0 - cond_probas[:, i])
    probas[:, -1] = cumprod[:, -1]
    return probas

def corn_label_from_logits(logits):
    return torch.argmax(corn_probas(logits), dim=1)

## Configuration

Everything tunable lives in this one cell. `PUBLISHED_ROOT` must contain `train/` and `val/`
subfolders, each holding `0/` .. `4/` grade folders of PNGs.

In [ ]:
# ---- reproducibility -------------------------------------------------------
SEED = 42

# ---- data ------------------------------------------------------------------
INPUT_SIZE = 384
ROTATION_DEGREES = 5

# ---- optimisation ----------------------------------------------------------
EPOCHS = 12
BATCH_SIZE = 48
NUM_WORKERS = 2
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-3
PRETRAINED = True          # ImageNet initialisation for the base run

# ---- paths -----------------------------------------------------------------
PUBLISHED_ROOT = Path(
    "/content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/"
    "extracted/KneeXrayData/ClsKLData/kneeKL224"
)
MODEL_ROOT = Path("/content/drive/MyDrive/Models/seresnext50_32x4d_original_focal_corn")

RUN_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y-%m-%d_%H-%M-%S_%f_UTC")
RUN_DIR = MODEL_ROOT / RUN_TIMESTAMP

if not PUBLISHED_ROOT.exists():
    raise FileNotFoundError(PUBLISHED_ROOT)
RUN_DIR.mkdir(parents=True, exist_ok=False)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:   ", DEVICE)
print("Run dir:  ", RUN_DIR)

## Index the published images

One row per image. Only `train` and `val` are indexed — the test split is reserved for Notebook 03.

In [4]:
rows = []
for split in ("train", "val"):
    for grade in range(5):
        for path in sorted((PUBLISHED_ROOT / split / str(grade)).glob("*.png")):
            rows.append({"split": split, "grade": grade, "image_path": str(path)})

frame = pd.DataFrame(rows)
if frame.empty:
    raise RuntimeError(f"No PNGs found under {PUBLISHED_ROOT}")
print(frame.groupby(["split", "grade"]).size().unstack(fill_value=0))

grade     0     1     2    3    4
split                            
train  2286  1046  1516  757  173
val     328   153   212  106   27


## Preprocessing, dataset, and model

The validation transform is byte-identical to the one the API applies at inference time. Keeping
them in sync is what makes the reported metrics representative of the deployed service.

In [ ]:
class OpenCVCLAHE:
    """LAB-space CLAHE. Identical to app/services/preprocessing_service.py."""

    def __call__(self, image_rgb):
        lab = cv2.cvtColor(np.asarray(image_rgb), cv2.COLOR_RGB2LAB)
        lightness, a, b = cv2.split(lab)
        lightness = cv2.createCLAHE(clipLimit=1.25, tileGridSize=(8, 8)).apply(lightness)
        return cv2.cvtColor(cv2.merge((lightness, a, b)), cv2.COLOR_LAB2RGB)


class SquarePad:
    """Pad to square with black borders, preserving aspect ratio."""

    def __call__(self, image_rgb):
        image = np.asarray(image_rgb)
        height, width = image.shape[:2]
        side = max(height, width)
        top, left = (side - height) // 2, (side - width) // 2
        return cv2.copyMakeBorder(
            image, top, side - height - top, left, side - width - left,
            cv2.BORDER_CONSTANT, value=(0, 0, 0),
        )


train_transform = transforms.Compose([
    OpenCVCLAHE(), SquarePad(), transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(p=0.50),
    transforms.RandomRotation(ROTATION_DEGREES),
    transforms.ColorJitter(brightness=0.08, contrast=0.08),
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.ToTensor(),
    transforms.RandomErasing(p=0.10, scale=(0.02, 0.05), ratio=(0.5, 2.0), value=0),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    OpenCVCLAHE(), SquarePad(), transforms.ToPILImage(),
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


class ImageDataset(Dataset):
    def __init__(self, data, transform):
        self.data = data.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        row = self.data.iloc[index]
        image = cv2.imread(row.image_path, cv2.IMREAD_COLOR)
        if image is None:
            raise IOError(f"Cannot read image: {row.image_path}")
        return self.transform(cv2.cvtColor(image, cv2.COLOR_BGR2RGB)), int(row.grade)



class SEResNeXt50Model(nn.Module):
    """Focal CORN variant. num_classes=4 -> 4 ordinal thresholds for 5 KL grades."""
    ARCHITECTURE = "seresnext50_32x4d_linear_gradcam_ordinal"
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model(
            "seresnext50_32x4d", pretrained=PRETRAINED, num_classes=NUM_CLASSES - 1
        )
    @property
    def gradcam_target_layer(self):
        return self.backbone.layer4
    def forward(self, images):
        return self.backbone(images)
build_model = SEResNeXt50Model


## Train

One pass per epoch, then validate and keep the best checkpoint by selection score.

In [ ]:
def selection_score(qwk, macro_f1, macro_ap):
    """Single scalar used to pick a checkpoint. Same weighting as the reference runs."""
    return 0.55 * qwk + 0.30 * macro_f1 + 0.15 * macro_ap


def score_predictions(labels, probabilities):
    labels = np.asarray(labels)
    probabilities = np.asarray(probabilities)
    predictions = corn_label_from_logits(torch.as_tensor(probabilities)).numpy() if False else corn_label_from_logits(torch.as_tensor(np.log(np.clip(probabilities, 1e-9, 1.0)))).numpy()
    _, _, macro_f1, _ = precision_recall_fscore_support(
        labels, predictions, average="macro", zero_division=0
    )
    qwk = cohen_kappa_score(labels, predictions, weights="quadratic")
    macro_ap = average_precision_score(np.eye(5)[labels], probabilities, average="macro")
    return {
        "qwk": float(qwk),
        "macro_f1": float(macro_f1),
        "macro_ap": float(macro_ap),
        "selection": float(selection_score(qwk, macro_f1, macro_ap)),
    }


train_frame = frame[frame.split == "train"].copy()
val_frame = frame[frame.split == "val"].copy()

counts = np.bincount(train_frame.grade.to_numpy(), minlength=5)
weights = (1.0 / counts)[train_frame.grade.to_numpy()]
sampler = WeightedRandomSampler(
    torch.as_tensor(weights, dtype=torch.double), len(weights), replacement=True
)
train_loader = DataLoader(
    ImageDataset(train_frame, train_transform), batch_size=BATCH_SIZE, sampler=sampler,
    num_workers=NUM_WORKERS, pin_memory=True,
)
val_loader = DataLoader(
    ImageDataset(val_frame, val_transform), batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)

model = build_model().to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-7)
scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")


def validate():
    model.eval()
    labels, probabilities = [], []
    with torch.inference_mode():
        for images, batch_labels in val_loader:
            probs = corn_probas(model(images.to(DEVICE, non_blocking=True)).float())
            labels.extend(batch_labels.numpy())
            probabilities.extend(probs.cpu().numpy())
    return score_predictions(labels, probabilities)


best_score = -float("inf")
history = []
for epoch in range(1, EPOCHS + 1):
    model.train()
    loss_sum, samples = 0.0, 0
    for images, labels in tqdm(train_loader, desc=f"epoch {epoch}/{EPOCHS}"):
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=DEVICE.type == "cuda"):
            loss = focal_corn_loss(model(images), labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        loss_sum += loss.item() * len(labels)
        samples += len(labels)
    scheduler.step()

    metrics = validate()
    row = {"epoch": epoch, "train_loss": loss_sum / samples, **metrics}
    history.append(row)
    print(json.dumps(row, indent=2))

    if metrics["selection"] > best_score:
        best_score = metrics["selection"]
        torch.save({
            "model_state_dict": model.state_dict(),
            "architecture": build_model.ARCHITECTURE,
            "loss_type": "focal_corn",
            "epoch": epoch,
            "input_size": INPUT_SIZE,
            "selection": best_score,
        }, RUN_DIR / "best_model.pth")
        print(f"  => new best, selection={best_score:.4f}")

pd.DataFrame(history).to_csv(RUN_DIR / "history.csv", index=False)

## Publish the checkpoint pointer

Notebook 02 reads `SELECTED_CHECKPOINT.txt` instead of a hard-coded timestamp, so re-running this
notebook automatically feeds the newer checkpoint forward.

In [ ]:
checkpoint_path = RUN_DIR / "best_model.pth"
(RUN_DIR / "SELECTED_CHECKPOINT.txt").write_text(str(checkpoint_path))
(RUN_DIR / "run_config.json").write_text(json.dumps({
    "stage": "01_original",
    "input_size": INPUT_SIZE,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "loss": "focal_corn",
    "best_selection": best_score,
}, indent=2))

print("Best checkpoint:", checkpoint_path)
print("Pointer written:", RUN_DIR / "SELECTED_CHECKPOINT.txt")
print("Next: run 02_train_paired_roi.ipynb")